In [4]:
%pip install tatc
from tatc import utils
from tatc.schemas import PointedInstrument, Satellite, Instrument, TwoLineElements

earthcare = Satellite(
    name = "EarthCare",
    orbit = TwoLineElements(
        tle=[
            "1 59908U 24101A   25200.34125573  .00010433  00000+0  14571-3 0  9999",
            "2 59908  97.0168 326.4971 0001222 108.6708 251.4681 15.57041891 64775"
        ]
    ),
    instruments=[
        PointedInstrument(
            name = "MSI",
            field_of_regard=utils.swath_width_to_field_of_regard(394e3,150e3) + 2*5.760868, #cross_track_field_of_view + 2*roll angle
            cross_track_field_of_view = utils.swath_width_to_field_of_view(394e3, 150e3, 5.760868),
            along_track_field_of_view = utils.swath_width_to_field_of_view(394e3, 10e3, 5.760868),
            roll_angle = 5.760868,  # degrees
            is_rectangular = True
        )
    ]
)

satellites = [earthcare]


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from datetime import datetime, timezone, timedelta
from dateutil.relativedelta import relativedelta
from tatc.analysis import compute_ground_track
import pandas as pd
from joblib import Parallel, delayed

startdate = datetime(2025,7,19,15,4, tzinfo = timezone.utc)  # initial date, discard the year.
duration = timedelta(days = 2*365)  # 2 years of satellite simulation
step = timedelta(seconds = 5)
batch_duration = timedelta(minutes=10)

def compute_groundtrack(satellite, start, duration, batch_duration, time_step):
    return pd.concat(
    Parallel(-1)(
        delayed(compute_ground_track)(
            satellite,
            pd.date_range(start + i*batch_duration, start + (i+1)*batch_duration, freq=time_step, inclusive="left"),
            crs="spice"
        )
        for i in range( duration // batch_duration)
        for satellite in satellites
    ),
    ignore_index=True
)

ground_tracks_high_res = compute_groundtrack(satellites[0], startdate, duration, batch_duration, step)

In [6]:
print(ground_tracks_high_res.head())
print(ground_tracks_high_res.columns)
print(len(ground_tracks_high_res))

                                            geometry  \
0  MULTIPOLYGON Z (((-22.06966 -23.90665 0, -22.0...   
1  MULTIPOLYGON Z (((-34.42459 -62.17122 0, -34.4...   
2  MULTIPOLYGON Z (((-131.76963 -83.07322 0, -131...   
3  MULTIPOLYGON Z (((-175.39874 -76.51405 0, -175...   
4  MULTIPOLYGON Z (((152.13363 -10.24487 0, 152.1...   

                       time  satellite instrument  valid_obs  
0 2025-07-19 15:04:00+00:00  EarthCare        MSI       True  
1 2025-07-19 15:14:00+00:00  EarthCare        MSI       True  
2 2025-07-19 15:24:00+00:00  EarthCare        MSI       True  
3 2025-07-19 15:34:00+00:00  EarthCare        MSI       True  
4 2025-07-19 15:44:00+00:00  EarthCare        MSI       True  
Index(['geometry', 'time', 'satellite', 'instrument', 'valid_obs'], dtype='object')
105120


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from cartopy import crs as ccrs
from IPython.display import HTML
from matplotlib.patches import Patch

plt.rcParams["animation.embed_limit"] = 500.0  # MB - accommodates 2-year span

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

frame_duration = batch_duration

def animate(frame):
    ax.clear()
    time = startdate + frame*frame_duration
    tracks = ground_tracks_high_res[
        (time <= ground_tracks_high_res.time)
        & (ground_tracks_high_res.time < time + frame_duration)
    ]
    if not tracks.empty:
        tracks.plot(ax=ax, color="r", transform=ccrs.PlateCarree())

    ax.set_global()
    ax.set_aspect("equal")
    ax.coastlines()
    ax.set_title(time)
    fig.tight_layout()

ani = animation.FuncAnimation(
    fig,
    animate,
    frames=duration // frame_duration,
    interval=100,
    blit=False
)
display(HTML(ani.to_jshtml()))
plt.close()

In [7]:
grid_size = 0.0625
g5nr_frame_duration = timedelta(minutes=30)  # native cadence of tavg30mn dataset

In [8]:
import rioxarray
import xarray as xr

dataset_high_res = xr.open_dataset(
    "https://opendap.nccs.nasa.gov/dods/OSSE/G5NR/Ganymed/7km/0.0625_deg/tavg/tavg30mn_2d_met3_Nx",
    decode_times=True,
)
dataset_high_res.rio.write_crs("epsg:4326", inplace=True)
dataset_high_res.rio.set_spatial_dims("lon", "lat", inplace=True)
dataset_high_res

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/xarray/coding/times.py:213: SerializationWarning: Ambiguous reference date string: 1-1-1 00:00:0.0. The first value is assumed to be the year hence will be padded with zeros to remove the ambiguity (the padded reference date string is: 0001-1-1 00:00:0.0). To remove this message, remove the ambiguity by padding your reference date strings with zeros.
  ref_date = _ensure_padded_year(ref_date)


<xarray.Dataset> Size: 138TB
Dimensions:      (time: 36576, lat: 2881, lon: 5760)
Coordinates:
  * time         (time) datetime64[ns] 293kB 2005-05-15T21:15:00 ... 2007-06-...
  * lat          (lat) float64 23kB -90.0 -89.94 -89.88 ... 89.88 89.94 90.0
  * lon          (lon) float64 46kB -180.0 -179.9 -179.9 ... 179.8 179.9 179.9
    spatial_ref  int64 8B 0
Data variables: (12/57)
    swgdnclr     (time, lat, lon) float32 2TB ...
    lwgabclrcln  (time, lat, lon) float32 2TB ...
    precsno      (time, lat, lon) float32 2TB ...
    ttauss       (time, lat, lon) float32 2TB ...
    prevtot      (time, lat, lon) float32 2TB ...
    cldhgh       (time, lat, lon) float32 2TB ...
    ...           ...
    tauhgh       (time, lat, lon) float32 2TB ...
    swtdn        (time, lat, lon) float32 2TB ...
    tsalt        (time, lat, lon) float32 2TB ...
    swgdn        (time, lat, lon) float32 2TB ...
    cldmid       (time, lat, lon) float32 2TB ...
    mxdiam       (time, lat, lon) float32 2TB ...
Attributes:
    title:                2d,30-Minute,Time-Averaged,Single-Level,Full Resolu...
    Conventions:          COARDS\nGrADS
    dataType:             Grid
    history:              Wed May 13 19:26:42 GMT 2026 : imported by GrADS Da...
    extra_das_attribute:  This is an example of metadata added using a supple...

In [9]:
# decorate ground_tracks_high_res with tautot from g5nr at each row's
# centroid+time. dedup-by-frame so OPeNDAP sees one batched slice fetch
# per time chunk instead of one point-style fetch per row -- 105k rows
# collapse to ~48 unique g5nr times under the (day=20, month=5, year=2006)
# wrap that matches the get_clusters pattern in the next cell.
import time as _time
import warnings
import gc
import numpy as np


IO_CHUNK_TIMES = 2   # ~130 MB per fetch at 0.0625deg; small to dodge OOM/SIGKILL
IO_RETRIES     = 3

def _frame_ds_time(frame):
    return (startdate + frame * g5nr_frame_duration).replace(
        day=20, month=5, year=2006, tzinfo=None
    )

def lookup_tautot(ground_tracks, dataset, startdate, g5nr_frame_duration):
    frames = ((ground_tracks.time - startdate) // g5nr_frame_duration).astype(int)
    # use geographic centroids (same as original code). the resulting error
    # is well below the 0.0625deg (~7 km) grid cell, so suppress the warning
    # rather than pay the cost of a full reprojection round-trip on 105k geoms.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        lats = ground_tracks.geometry.centroid.y.values
        lons = ground_tracks.geometry.centroid.x.values

    unique_frames = sorted(frames.unique())
    frame_to_time = {int(f): _frame_ds_time(int(f)) for f in unique_frames}
    unique_times  = sorted(set(frame_to_time.values()))
    chunks = [unique_times[i:i + IO_CHUNK_TIMES]
              for i in range(0, len(unique_times), IO_CHUNK_TIMES)]
    print(f"{len(ground_tracks)} rows -> {len(unique_times)} distinct g5nr times "
          f"-> {len(chunks)} OPeNDAP chunks", flush=True)

    out = np.full(len(ground_tracks), np.nan, dtype=np.float32)
    _t0 = _time.time()
    for ci, chunk_times in enumerate(chunks):
        _tc = _time.time()
        t_min, t_max = min(chunk_times), max(chunk_times)
        sub = None
        for attempt in range(IO_RETRIES):
            try:
                sub = (
                    dataset["tautot"]
                    .sel(time=slice(t_min, t_max), lat=slice(-89, 89))
                    .load()
                )
                break
            except Exception as e:
                if attempt == IO_RETRIES - 1:
                    raise
                print(f"  chunk {ci+1} attempt {attempt+1} failed: {e!r}; retrying",
                      flush=True)
                _time.sleep(2 ** attempt)
        chunk_set = set(chunk_times)
        for f in unique_frames:
            t = frame_to_time[int(f)]
            if t not in chunk_set:
                continue
            mask = (frames == f).values
            out[mask] = (
                sub.sel(time=t, method="nearest")
                   .sel(lat=xr.DataArray(lats[mask], dims="p"),
                        lon=xr.DataArray(lons[mask], dims="p"),
                        method="nearest")
                   .values
            )
        # explicit drop so the next iteration's .load() doesn't sit alongside
        # this one in memory while xarray's refcount stabilizes
        del sub
        gc.collect()
        print(f"chunk {ci+1}/{len(chunks)}: {len(chunk_times)} times, "
              f"{_time.time()-_tc:.1f}s", flush=True)
    print(f"total: {_time.time()-_t0:.1f}s", flush=True)
    return out

ground_tracks_high_res["tautot"] = lookup_tautot(
    ground_tracks_high_res, dataset_high_res, startdate, g5nr_frame_duration
)
display(ground_tracks_high_res.head())


105120 rows -> 48 distinct g5nr times -> 24 OPeNDAP chunks
chunk 1/24: 2 times, 57.9s
chunk 2/24: 2 times, 28.1s
chunk 3/24: 2 times, 30.2s
chunk 4/24: 2 times, 32.3s
chunk 5/24: 2 times, 30.8s
chunk 6/24: 2 times, 30.8s
chunk 7/24: 2 times, 61.7s
chunk 8/24: 2 times, 85.4s
chunk 9/24: 2 times, 68.6s
chunk 10/24: 2 times, 64.2s
chunk 11/24: 2 times, 62.0s
chunk 12/24: 2 times, 73.8s
chunk 13/24: 2 times, 71.8s
chunk 14/24: 2 times, 67.9s
chunk 15/24: 2 times, 84.1s
chunk 16/24: 2 times, 60.2s
chunk 17/24: 2 times, 62.9s
chunk 18/24: 2 times, 80.2s
chunk 19/24: 2 times, 80.9s
chunk 20/24: 2 times, 67.5s
chunk 21/24: 2 times, 116.1s
chunk 22/24: 2 times, 71.1s
chunk 23/24: 2 times, 54.9s
chunk 24/24: 2 times, 72.4s
total: 1515.9s


,geometry,time,satellite,instrument,valid_obs,tautot
0,"MULTIPOLYGON Z (((-22.06966 -23.90665 0, -22.0...",2025-07-19 15:04:00+00:00,EarthCare,MSI,True,6.078127
1,"MULTIPOLYGON Z (((-34.42459 -62.17122 0, -34.4...",2025-07-19 15:14:00+00:00,EarthCare,MSI,True,18.843752
2,"MULTIPOLYGON Z (((-131.76963 -83.07322 0, -131...",2025-07-19 15:24:00+00:00,EarthCare,MSI,True,0.000000
3,"MULTIPOLYGON Z (((-175.39874 -76.51405 0, -175...",2025-07-19 15:34:00+00:00,EarthCare,MSI,True,7.554689
4,"MULTIPOLYGON Z (((152.13363 -10.24487 0, 152.1...",2025-07-19 15:44:00+00:00,EarthCare,MSI,True,11.984377


In [10]:
import numpy as np
import geopandas as gpd
from shapely.geometry import box
import scipy.ndimage as ndi

def compute_grid_cell_area(lat, lon):
    R = 6371.0  # Earth radius in km
    d2r = np.pi / 180.0
    ny, nx = len(lat), len(lon)

    dlat = abs(lat[1] - lat[0]) if len(lat) > 1 else 1.0
    dlon = abs(lon[1] - lon[0]) if len(lon) > 1 else 1.0

    lat_rad = lat * d2r
    dlat_rad = dlat * d2r
    dlon_rad = dlon * d2r

    area = np.zeros((ny, nx), dtype=np.float64)
    for i in range(ny):
        area[i, :] = (
            (R**2) * dlon_rad *
            (np.sin(lat_rad[i] + dlat_rad / 2) - np.sin(lat_rad[i] - dlat_rad / 2))
        )
    return area

In [11]:
import scipy.ndimage as ndi
import numpy as np
import geopandas as gpd
import pandas as pd
import xarray as xr
import time as _time
from shapely.geometry import box
from joblib import Parallel, delayed

# parallelized get_clusters: dedup frame -> g5nr-time, batched OpenDap
# fetch per chunk, then per-frame CPU work in threads.

threshold = 220  # brightness temperature threshold in Kelvin

# tunables - IO_CHUNK_TIMES dropped to 8 because each 0.0625deg slice is
# ~66 MB per variable, so 8 slices x 2 vars approx 1 GB per OpenDap request.
IO_CHUNK_TIMES = 8
IO_RETRIES     = 3
CPU_WORKERS    = -1


def _frame_ds_time(frame):
    # same time-mapping trick the 0.5deg notebook uses
    return (startdate + frame * g5nr_frame_duration).replace(
        day=20, month=5, year=2006, tzinfo=None
    )


def _fetch_chunk(chunk_times):
    # one batched HTTP request via a contiguous time slice, then local pick
    t_min, t_max = min(chunk_times), max(chunk_times)
    for attempt in range(IO_RETRIES):
        try:
            sub = (
                dataset_high_res[["lwtup", "prectot"]]
                .sel(time=slice(t_min, t_max), lat=slice(-89, 89))
                .load()
            )
            break
        except Exception:
            if attempt == IO_RETRIES - 1:
                raise
            _time.sleep(2 ** attempt)
    pick = xr.DataArray(pd.to_datetime(chunk_times).values, dims="t")
    sub = sub.sel(time=pick, method="nearest")
    return {
        t: (sub["lwtup"].isel(t=j).values, sub["prectot"].isel(t=j).values)
        for j, t in enumerate(chunk_times)
    }


def _process_frame(frame, arrays, area_2d, lat2d, lon2d):
    # CPU-only: build the dissolved cluster GeoDataFrame for one frame
    try:
        lwtup_2d, prectot_2d = arrays[_frame_ds_time(frame)]

        tb = np.sqrt(np.sqrt(lwtup_2d / 5.67037e-8))
        labels, _ = ndi.label(tb < threshold)

        mask = labels > 0
        if not mask.any():
            return gpd.GeoDataFrame()

        cluster_labels = labels[mask]
        lat_vals       = lat2d[mask]
        lon_vals       = lon2d[mask]
        prectot_vals   = prectot_2d[mask]
        area_vals      = area_2d[mask]

        # single NaN filter across all columns - keeps lengths aligned
        finite = ~np.isnan(prectot_vals)
        cluster_labels = cluster_labels[finite]
        lat_vals       = lat_vals[finite]
        lon_vals       = lon_vals[finite]
        prectot_vals   = prectot_vals[finite]
        area_vals      = area_vals[finite]

        if cluster_labels.size == 0:
            return gpd.GeoDataFrame()

        cells = gpd.GeoDataFrame(
            {
                "count": 1,
                "cluster": cluster_labels,
                "lat": lat_vals,
                "lon": lon_vals,
                "time": startdate + frame * g5nr_frame_duration,
                "prectot": prectot_vals,
                "area": area_vals,
            },
            geometry=[
                box(lo, la, lo + grid_size, la + grid_size)
                for lo, la in zip(lon_vals, lat_vals)
            ],
            crs="EPSG:4326",
        )
        cells["tot_prectot"] = cells["prectot"]
        cells["avg_prectot"] = cells["prectot"]
        cells["max_prectot"] = cells["prectot"]

        return cells[cells.cluster > 0].dissolve(
            by=["time", "cluster"],
            aggfunc={
                "count": "sum",
                "area": "sum",
                "tot_prectot": "sum",
                "avg_prectot": "mean",
                "max_prectot": "max",
            },
        )
    except Exception as e:
        print(f"Frame {frame} failed: {e}")
        return gpd.GeoDataFrame()


_t0 = _time.time()
n_frames = int(duration // g5nr_frame_duration)

# precompute area + meshgrid once, shared across every frame
sample_lat = dataset_high_res["lat"].sel(lat=slice(-89, 89)).values
sample_lon = dataset_high_res["lon"].values
area_2d = compute_grid_cell_area(sample_lat, sample_lon)
lon2d, lat2d = np.meshgrid(sample_lon, sample_lat)

# dedup: many frames map to the same g5nr timestamp under the .replace mapping
unique_times = sorted({_frame_ds_time(f) for f in range(n_frames)})
print(f"{n_frames} frames -> {len(unique_times)} distinct g5nr time slices")

# group frames by which time-chunk their data lives in (bounds memory)
time_chunks = [
    unique_times[i:i + IO_CHUNK_TIMES]
    for i in range(0, len(unique_times), IO_CHUNK_TIMES)
]
time_to_chunk_idx = {t: ci for ci, c in enumerate(time_chunks) for t in c}
frames_by_chunk = [[] for _ in time_chunks]
for f in range(n_frames):
    frames_by_chunk[time_to_chunk_idx[_frame_ds_time(f)]].append(f)

# stream: fetch chunk -> process its frames in parallel -> drop arrays
all_results = []
for ci, chunk_times in enumerate(time_chunks):
    _tc = _time.time()
    arrays = _fetch_chunk(chunk_times)
    chunk_results = Parallel(n_jobs=CPU_WORKERS, backend="threading")(
        delayed(_process_frame)(f, arrays, area_2d, lat2d, lon2d)
        for f in frames_by_chunk[ci]
    )
    all_results.extend(chunk_results)
    print(f"chunk {ci+1}/{len(time_chunks)}: "
          f"{len(chunk_times)} times, {len(frames_by_chunk[ci])} frames, "
          f"{_time.time()-_tc:.1f}s")

clusters_high_res = pd.concat([r for r in all_results if not r.empty]).reset_index()
print(f"total: {_time.time()-_t0:.1f}s, {len(clusters_high_res)} cluster rows")
display(clusters_high_res)

35040 frames -> 48 distinct g5nr time slices


: 

In [ ]:
from skyfield.api import wgs84
from tatc.constants import de421, timescale

# compute solar hour based on the angle of the sun as seen at the feature centroid
clusters_high_res["solar_hour"] = clusters_high_res.apply(
    lambda r: (
        de421["earth"] + wgs84.latlon(r.geometry.centroid.y, r.geometry.centroid.x)
    )
    .at(timescale.from_datetime(r.time + frame_duration/2))
    .observe(de421["sun"])
    .apparent()
    .hadec()[0]
    .hours
    + 12,
    axis=1,
)
display(clusters_high_res)

In [ ]:
import shapely

# bin times to the g5nr frame grid that both clusters_high_res and
# ground_tracks_high_res live on. this is the same
# (time - startdate) // g5nr_frame_duration trick used in lookup_tautot
# and in get_clusters, so the bin a cluster row falls in is exactly the
# bin its observation window (30 min here, native tavg30mn cadence)
# covers — the per-bin dissolve below is semantically equivalent to the
# original per-row apply, but pays the unary_union cost once per bin
# instead of once per cluster row.
clusters_hbin = ((clusters_high_res.time      - startdate) // g5nr_frame_duration).astype(int)
gt_hbin       = ((ground_tracks_high_res.time - startdate) // g5nr_frame_duration).astype(int)

# dissolve ground_tracks_high_res once per frame bin (one union geom per bin)
gt_per_hbin = ground_tracks_high_res.assign(hbin=gt_hbin).dissolve(by="hbin")[["geometry"]]

# left-merge the per-bin union onto each cluster row.
# suffixes=("", "_gt") keeps the active geometry column intact and puts the
# ground-track union under "geometry_gt".
merged = clusters_high_res.assign(hbin=clusters_hbin).merge(
    gt_per_hbin, left_on="hbin", right_index=True, how="left",
    suffixes=("", "_gt"),
)

# vectorized intersects via shapely 2.x.
# rows whose bin has no ground-track passes get None on the right and stay 0.
left_geom  = merged["geometry"].values
right_geom = merged["geometry_gt"].values
has_gt   = np.array([g is not None for g in right_geom])
observed = np.zeros(len(clusters_high_res), dtype=int)
observed[has_gt] = shapely.intersects(left_geom[has_gt], right_geom[has_gt]).astype(int)
clusters_high_res["observed"] = observed

display(clusters_high_res)
print(clusters_high_res[clusters_high_res["observed"] == 1], "features observed")


In [ ]:
import matplotlib.animation as animation
from cartopy import crs as ccrs
from IPython.display import HTML

plt.rcParams["animation.embed_limit"] = 500.0  # MB - accommodates 2-year span

# create a figure
fig, ax = plt.subplots(figsize=(8,4), subplot_kw={"projection": ccrs.PlateCarree()})

def animate(frame):
    ax.clear()
    time = startdate + frame*g5nr_frame_duration
    active_clusters = clusters_high_res[
        (clusters_high_res.time >= time)
        & (clusters_high_res.time < time + g5nr_frame_duration)
    ]
    if not active_clusters.empty:
        active_clusters.plot(
            ax=ax,
            column="observed",
            vmin=0,
            vmax=1,
            cmap="RdYlGn"
        )
    track = ground_tracks_high_res[
        (ground_tracks_high_res.time >= time)
        & (ground_tracks_high_res.time < time + g5nr_frame_duration)
    ]
    if not track.empty:
        track.dissolve().plot(ax=ax, color="black", alpha = 0.2)
    ax.set_global()
    ax.set_aspect("equal")
    ax.coastlines()
    fig.tight_layout()
    plt.legend(handles=[
        Patch(label="MSI", facecolor="black", alpha=0.8),
        Patch(label="Unobserved", facecolor="#A50026"),
        Patch(label="Observed", facecolor="#006837"),
    ], loc="lower right")

ani = animation.FuncAnimation(
    fig,
    animate,
    frames=duration//g5nr_frame_duration,
    interval=100,
    blit=False
)
display(HTML(ani.to_jshtml()))
plt.close()

In [ ]:
# observed-only join: each row is a 10-minute satellite pass that scooped
# at least one cluster the satellite was looking at, with prectot stats
# from those clusters and tautot stats from the track itself.

# define a frame number for both datasets — used by the sjoin's
# on_attribute="frame" hour/half-hour-binned matching.
ground_tracks_high_res["frame"] = (ground_tracks_high_res.time - startdate) // g5nr_frame_duration
clusters_high_res["frame"]      = (clusters_high_res.time      - startdate) // g5nr_frame_duration

observed_clusters_high_res = clusters_high_res[clusters_high_res["observed"] == 1].copy()

# cluster-centroid lat/lon — added so the agg below can produce mean lat/lon
# per ground_track group. uses centroids in geographic CRS (EPSG:4326),
# which emits a benign UserWarning; the resulting error is well below the
# 0.5/0.0625-degree grid cell size for ITCZ-band analysis.
observed_clusters_high_res["lat"] = observed_clusters_high_res.geometry.centroid.y
observed_clusters_high_res["lon"] = observed_clusters_high_res.geometry.centroid.x

joined_observed_high_res = gpd.sjoin(
    observed_clusters_high_res, ground_tracks_high_res,
    how="right",
    predicate="intersects",
    on_attribute="frame",
)

# Drop the ground_track rows that didn't intersect any observed cluster
# BEFORE the agg. Doing this post-agg via dropna(subset=["tot_prectot"]) does
# NOT work in modern pandas: groupby().sum() defaults to min_count=0, so an
# all-NaN group sums to 0 (not NaN) — indistinguishable from a real-zero
# group — and the post-agg dropna can't filter them out. Pre-filtering
# removes the empty groups entirely so the row count of aggregated_observed
# becomes "ground_tracks that scored >=1 observed cluster".
matched_high_res = joined_observed_high_res.dropna(subset=["tot_prectot"])

# Cast to a plain pandas DataFrame BEFORE the agg. The agg's internal concat
# (concat({col: result, ...}, axis=1, keys=keys_to_use)) builds MultiIndex
# columns temporarily, then calls __finalize__ on the result. When the input
# was a GeoDataFrame, geopandas's __finalize__ runs the safety check
# `(self.columns == self._geometry_column_name).sum() > 1` — and comparing a
# MultiIndex of tuples to the scalar "geometry" raises
# 'ValueError: truth value of an array with more than one element is ambiguous'.
# Passing a plain DataFrame skips that check; we re-wrap as a GeoDataFrame after.
joined_df_high_res = pd.DataFrame(matched_high_res)

_agg_high_res = joined_df_high_res.groupby(level=0).agg(
    geometry    =("geometry",    "first"),
    time        =("time_left",   "first"),
    solar_hour  =("solar_hour",  "first"),
    satellite   =("satellite",   "first"),
    count       =("count",       "sum"),
    area        =("area",        "sum"),
    tot_prectot =("tot_prectot", "sum"),
    avg_prectot =("avg_prectot", "mean"),
    max_prectot =("max_prectot", "max"),
    tot_tautot  =("tautot",      "sum"),
    avg_tautot  =("tautot",      "mean"),
    max_tautot  =("tautot",      "max"),
    lat         =("lat",         "mean"),
    lon         =("lon",         "mean"),
)

aggregated_observed_high_res = gpd.GeoDataFrame(
    _agg_high_res, geometry="geometry", crs=ground_tracks_high_res.crs,
)

display(aggregated_observed_high_res)
print(len(aggregated_observed_high_res), "track rows joined with observed clusters")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.colors as mcolors
from cartopy import crs as ccrs
from IPython.display import HTML

plt.rcParams["animation.embed_limit"] = 500.0  # MB - accommodates 2-year span

fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

frame_duration = batch_duration

prectot_vmin = aggregated_observed_high_res['avg_prectot'].min()
prectot_vmax = aggregated_observed_high_res['avg_prectot'].max()

# first plot to generate a colorbar
aggregated_observed_high_res.plot(
    ax=ax,
    column="avg_prectot",
    norm=mcolors.LogNorm(prectot_vmin, prectot_vmax),
    transform=ccrs.PlateCarree(),
    legend=True,
    legend_kwds={"orientation": "horizontal", "label": "Average Precipitation (kg/m$^2$/s)"}
)

def animate(frame):
    ax.clear()
    time = startdate + frame*frame_duration
    tracks = aggregated_observed_high_res[
        (aggregated_observed_high_res.time >= time)
        & (aggregated_observed_high_res.time < time + frame_duration)
    ]
    if not tracks.empty:
        tracks.boundary.plot(ax=ax, color="r", linewidth=0.5)
        tracks.plot(
            ax=ax,
            column="avg_prectot",
            norm=mcolors.LogNorm(prectot_vmin, prectot_vmax),
            transform=ccrs.PlateCarree(),
        )
    ax.set_global()
    ax.set_aspect("equal")
    ax.coastlines()
    ax.set_title(time)
    fig.tight_layout()

ani = animation.FuncAnimation(
    fig,
    animate,
    frames=duration // frame_duration,
    interval=100,
    blit=False
)
display(HTML(ani.to_jshtml()))
plt.close()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

clusters_filtered_high_res = clusters_high_res[
    (clusters_high_res['time'].dt.day == 19)
]
hourly_stats = (
    clusters_filtered_high_res[:1100]
    .groupby('solar_hour')
    .agg(mean_prectot=('avg_prectot', 'mean'))
    .reset_index()
    .sort_values('solar_hour')
)

print(hourly_stats.head())
print(len(hourly_stats))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hourly_stats['solar_hour'], hourly_stats['mean_prectot'], marker = '.', color = 'b')
ax.set_xlabel('Solar Hour (0–23)')
ax.set_ylabel('AVG_PRECTOT (kg m⁻² s⁻¹)')
ax.set_title('Mean Precipitation vs Solar Hour (0.0625°, 2 years)')
ax.set_xticks(range(0, 25, 2))
plt.tight_layout()
plt.show()

#probability distribution (KDE)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))

sns.kdeplot(
    data=clusters_filtered_high_res[:1100],
    x="solar_hour",
    y="avg_prectot",
    fill=True,
    cmap="Reds",
    bw_adjust=0.5,
    ax=ax,
)

ax.set_xlabel("Solar Hour (0–24)")
ax.set_ylabel("AVG_PRECTOT (kg m⁻² s⁻¹)")
ax.set_title("Mean Precipitation vs Solar Hour (0.0625°, 2 years)")
ax.set_xlim(0, 23)
ax.set_xticks(range(0, 25, 2))
ax.set_ylim(0, 4e-3)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import pandas as pd

agg = aggregated_observed_high_res.copy()
agg["centroid"] = agg.geometry.centroid

flat = pd.DataFrame({
    "lon": agg.centroid.x,
    "lat": agg.centroid.y,
    "avg_prectot": agg.avg_prectot,
    "tot_prectot": agg.tot_prectot,
    "time": agg.time,
})

fig, ax = plt.subplots(figsize=(10, 5), subplot_kw={"projection": ccrs.PlateCarree()})

sns.kdeplot(
    data=flat,
    x="lon",
    y="lat",
    weights="avg_prectot",
    fill=True,
    cmap="turbo",
    bw_adjust=0.2,
    ax=ax
)

ax.coastlines()
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_title("KDE of Simulated Mean Precipitation observable by EarthCare MSI for 2 years (0.0625°)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xticks(range(-179, 179, 30))
ax.set_yticks(range(-90, 91, 30))
ax.gridlines(draw_labels=True)
plt.show()

In [ ]:
import seaborn as sns
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import pandas as pd

agg = aggregated_observed_high_res.copy()
agg["centroid"] = agg.geometry.centroid

flat = pd.DataFrame({
    "lon": agg.centroid.x,
    "lat": agg.centroid.y,
    "avg_tautot": agg.avg_tautot,
    "tot_tautot": agg.tot_tautot,
    "time": agg.time,
})

fig, ax = plt.subplots(figsize=(10, 5), subplot_kw={"projection": ccrs.PlateCarree()})

sns.kdeplot(
    data=flat,
    x="lon",
    y="lat",
    weights="avg_tautot",
    fill=True,
    cmap="turbo",
    bw_adjust=0.2,
    ax=ax
)

ax.coastlines()
ax.set_xlim(-180, 180)
ax.set_ylim(-90, 90)
ax.set_title("KDE of Average Cloud Cover Measured by EarthCare MSI for 2 years (0.0625°)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_xticks(range(-179, 179, 30))
ax.set_yticks(range(-90, 91, 30))
ax.gridlines(draw_labels=True)
plt.show()

In [ ]:
# sanity check 1: are observed clusters spread reasonably across frames?
# expectation: ~total_observed / n_frames per frame on average, with a
# reasonable spread. a giant pile-up at zero or a few frames hoarding the
# bulk would point to a binning bug in the vectorized observed rewrite.
obs = clusters_high_res[clusters_high_res["observed"] == 1]
hbin = ((obs["time"] - startdate) // g5nr_frame_duration).astype(int)
per_frame = obs.groupby(hbin).size()
print("observed clusters per g5nr frame:")
print(per_frame.describe())
print(f"\nframes containing at least one observed cluster: {len(per_frame)}")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(per_frame.values, bins=40)
ax.set_xlabel("observed clusters per frame")
ax.set_ylabel("count of frames")
ax.set_title("Per-frame distribution of observed clusters")
plt.tight_layout()
plt.show()


In [ ]:
# sanity check 2: latitude distribution of observed clusters.
# expectation for real deep convection: peak in the tropics (|lat| < 30°)
# with a secondary mode in mid-latitude storm tracks.
# EarthCARE's sun-sync orbit at ~97 deg inclination geometrically over-samples
# latitudes near +/- 82 deg (where ground tracks converge), but cold-cloud
# clusters concentrate in the tropics — so the *weighted* observed
# distribution should still skew tropical. heavy concentration above |lat|=60
# usually means cold polar surfaces (Antarctic ice, sea-ice edges) are being
# detected by the tb<220K threshold as if they were deep convective cloud tops.
obs = clusters_high_res[clusters_high_res["observed"] == 1]
lats = obs.geometry.centroid.y
print("centroid lat distribution of observed clusters:")
print(lats.describe())

bands = [
    ("south polar  (lat <= -75)",       lats <= -75),
    ("antarctic    (-75 < lat <= -60)", (lats > -75) & (lats <= -60)),
    ("S mid-lat    (-60 < lat <= -30)", (lats > -60) & (lats <= -30)),
    ("S tropics    (-30 < lat <= 0)",   (lats > -30) & (lats <= 0)),
    ("N tropics    ( 0 < lat <= 30)",   (lats >  0)  & (lats <= 30)),
    ("N mid-lat    (30 < lat <= 60)",   (lats > 30) & (lats <= 60)),
    ("arctic       (60 < lat <= 75)",   (lats > 60) & (lats <= 75)),
    ("north polar  (lat > 75)",          lats > 75),
]
print(f"\nobserved clusters by latitude band (total: {len(obs)}):")
for name, mask in bands:
    n = int(mask.sum())
    pct = 100 * n / max(len(obs), 1)
    print(f"  {name:<32}  {n:>7} ({pct:>5.1f}%)")

import matplotlib.pyplot as plt
import numpy as np
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(lats.values, bins=np.linspace(-90, 90, 37))
ax.axvspan(-23.5, 23.5, alpha=0.15, color="green", label="tropics (|lat|<23.5°)")
ax.set_xlabel("centroid latitude (deg)")
ax.set_ylabel("count of observed clusters")
ax.set_title("Latitude distribution of observed clusters")
ax.legend()
plt.tight_layout()
plt.show()
